# 03 · Add a small adapter
Keep the base weights fixed. Add a small set of weights that can learn the task.

**Frozen base + trainable LoRA adapter → adapted model**

| Component | Job in this lesson |
|---|---|
| **OpenClaw + lightweight Nano** | Read the skill and coordinate the model-building tools. |
| **Python + PEFT** | Create a tiny model, attach LoRA, and inspect the weights. |
| **Lightning** | Remains the workshop’s serving model. No Lightning inference or training is needed in this CPU exercise. |

This is a **tiny, randomly initialized practice model**, not a smaller pretrained Lightning. It makes the mechanics quick to inspect. We attach the adapter here; training comes in notebook 04.

###  parameter-efficient fine-tuning

[PEFT quicktour](https://huggingface.co/docs/peft/main/en/quicktour) for configuring and attaching adapters; [Transformers](https://huggingface.co/docs/transformers/index) for model construction. Our environment pins PEFT 0.21.2 and Transformers 5.19.0; upstream `main` pages can differ.

build the CPU base, attach rank 8, inspect the trainable parameter names, then compare ranks 4/8/16. Saving here writes the base and adapter recipe; training happens in 04. No model service or weight download is needed for the manual exercise.

Source: [lora_walkthrough.py](agent/lora_walkthrough.py) wraps `GPT2LMHeadModel`, `LoraConfig`, and `get_peft_model`. Inspect `make_base`, `attach_adapter`, and `save_setup` to see the SDK calls behind the displays.

the base is randomly initialized and tiny. This demonstrates adapter mechanics, not useful pretrained language knowledge. The adapter cannot be attached to Lightning or a hosted OpenAI model.


### Start here
Load the practice-model helpers. Everything in this lesson is created locally on the CPU; no model download is needed.

In [ ]:
from pathlib import Path
import sys, json
from IPython.display import JSON, display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "course/common.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course.agent.lora_walkthrough import (
    make_base, attach_adapter, show_parameters, compare_ranks, save_setup,
)

### 1 · Create the practice model
A model is a collection of layers and their weights. This one has **1 transformer block, 2 attention heads, and a width of 48**.

Run the cell to see its actual configuration.

In [ ]:
base = make_base()
print(type(base).__name__)
display(JSON({
    "layers": base.config.n_layer,
    "attention_heads": base.config.n_head,
    "hidden_width": base.config.n_embd,
    "parameters_before_adapter": sum(p.numel() for p in base.parameters()),
}, expanded=True))

### API pattern · attach a PEFT adapter

Adapted from the [PEFT quicktour](https://huggingface.co/docs/peft/main/en/quicktour) to this lesson's GPT-2 layers. Reference only: the next exercise cell calls our wrapper, so do not attach twice to the same base.

```python
from peft import LoraConfig, get_peft_model
config = LoraConfig(
    r=8, lora_alpha=16, target_modules=['c_attn', 'c_proj', 'c_fc'],
    task_type='CAUSAL_LM', fan_in_fan_out=True, lora_dropout=0.0,
)
adapted = get_peft_model(base, config)
```

Layer names depend on the model architecture. Do not reuse these targets for Lightning.


### 2 · Attach LoRA
Instead of updating a whole weight matrix, LoRA learns two smaller matrices, **A** and **B**.

**Layer output = frozen base output + small adapter update**

`rank` controls the size of those smaller matrices. We start at **8**. PEFT freezes the original weights and creates the trainable adapter weights.

Rerunning the next cell creates a fresh base first, so adapters do not stack up.

In [ ]:
rank = 8
model = attach_adapter(make_base(), rank=rank)
report = show_parameters(model)

### 3 · Inspect the weights that can learn
Run the cell to inspect the actual adapter matrix names and shapes. Names containing `lora_A` and `lora_B` belong to the adapter.

The check must say **only LoRA trainable: True**. These percentages describe this tiny practice model—not the production Lightning adapter.

In [ ]:
print("Only LoRA trainable:", report["only_lora_trainable"])
display(JSON(report["adapter_matrices"], expanded=True))

### 4 · Make rank visible
Compare ranks **4, 8, and 16**. Each row builds an independent adapter on the same practice architecture.

Higher rank adds more trainable parameters. It is a capacity choice, not a guarantee of better accuracy. This comparison does not change the adapter you created above.

In [ ]:
rank_comparison = compare_ranks([4, 8, 16])

### 5 · Save the starting point
Save two things for the next step:

| Artifact | What it contains |
|---|---|
| `base/` | The original, unadapted practice-model weights. |
| `adapter-config/adapter_config.json` | The recipe for creating the adapter: rank, scale, and target layers. |

These are **not trained adapter weights yet**. The helper checks that every base weight is still unchanged before saving.

In [ ]:
saved = save_setup(model, ROOT / "course/outputs/preview03/cpu")
display(JSON(saved, expanded=True))

In [ ]:
adapter_json = json.loads(Path(saved["adapter_config"]).read_text())
display(JSON(adapter_json, expanded=True))

**Read the recipe:** `r` is rank, `lora_alpha` sets the update scale, and `target_modules` selects which layer types receive adapters. Here the scale is `lora_alpha / r = 2`.

These target names belong to the practice GPT-2 architecture. Lightning uses different layer names.

### 6 · Grow the skill library
| Notebook | Skill | What it adds |
|---|---|---|
| 01 | `workshop-model-inventory` | Discover models and test responses. |
| 02 | `workshop-training-data` | Build and check labeled examples. |
| 03 | `workshop-build-lora` | Attach an adapter and check which weights can learn. |

The new skill gives OpenClaw three actions:

**`inspect-model` → `attach-lora` → `check-lora`**

Nano chooses the calls. The fixed Python tools build and inspect the practice model on the host CPU. They do not train or restart Lightning.

In [ ]:
from html import escape
from IPython.display import HTML
skill = (ROOT / "course/agent/skills/workshop-build-lora/SKILL.md").read_text()
display(HTML("<details><summary><b>Read SKILL.md</b></summary><pre style='white-space:pre-wrap'>"
             + escape(skill) + "</pre></details>"))

### 7 · Prepare the terminal demo
Run once for each new demo. This cell installs the growing skill library in a fresh sandbox workspace and starts a background host worker for this lesson’s three actions. **It does not ask the agent to do anything yet.**

Here, **installing a skill means placing its `SKILL.md` instruction file in OpenClaw’s workspace**. The current lesson’s source is `course/agent/skills/workshop-build-lora/SKILL.md` in this repository.

Each run gets this structure inside the OpenShell sandbox (launcher files omitted):

```text
/home/sandbox/course-runs/openclaw-<run-id>/
├── openclaw.json                 # points OpenClaw at this run’s workspace
├── chapter_tool.py              # helper invoked by the agent
├── chapter.json                 # this lesson’s task and allowed actions
└── workspace/
    ├── AGENTS.md                 # workspace instructions
    └── skills/
        ├── workshop-build-lora/
        │   └── SKILL.md
        └── ...                  # skills from earlier notebooks
```

OpenClaw discovers skills under `workspace/skills/`. The task you paste in the next step tells the agent which skill to read and supplies the full path to `chapter_tool.py` (the skill’s `<tool_path>` placeholder).

The agent invokes the helper through its execution tool. The helper forwards the selected action to the host worker, which runs the fixed lesson code and returns the result. Nano follows the skill and coordinates the calls.

This lesson does not require artifacts from an earlier OpenClaw demo.

Start the TUI task within **15 minutes** and keep this run’s `terminal_run` value for collecting results. If the worker expires, prepare a new run. The local run directory stores host results; the installed skills live at the sandbox paths printed below.


In [ ]:
from course.agent.chapter_terminal import prepare_chapter_terminal, collect_chapter_terminal
from course.agent.notebook_terminal import open_terminal
terminal_run = prepare_chapter_terminal('03-build-lora')
print("Local run directory:", terminal_run["directory"])
print("OpenClaw sandbox workspace:", terminal_run["remote"] + "/workspace")
print("Installed skill:", terminal_run["remote"] + "/workspace/skills/workshop-build-lora/SKILL.md")
print("Python helper:", terminal_run["remote"] + "/chapter_tool.py")
print("\nThe agent will run these commands individually:")
for action in ['inspect-model', 'attach-lora', 'check-lora']:
    print("python3 " + terminal_run["remote"] + "/chapter_tool.py " + action)

### 8 · Open OpenClaw and paste the task
Exit the previous TUI with `/exit` first. The next cell opens a terminal on the workshop computer’s desktop.

The model label says **Nano**, the lightweight orchestration model. Copy only the generated task below into the TUI. Watch it read the skill and call the three tools.

In [ ]:
open_terminal(terminal_run)
print("\nPASTE THIS INTO OPENCLAW:\n")
print(terminal_run["prompt"])

### 9 · Bring the results back
Wait until the TUI finishes, then run this cell. Look for **PASS**, **rank 8**, the frozen and trainable parameter counts, and **only_lora_trainable: true**.

This checks the agent’s own artifacts and verifies that it read the skill before executing tools. Running all cells without completing the TUI task will stop here; rerun this cell afterward.

In [ ]:
agent_result = collect_chapter_terminal(terminal_run)
display(JSON(agent_result, expanded=True))

**What you demonstrated:** create a base → attach LoRA → see which weights can learn → compare rank → let Nano coordinate the same work.

**Next:** notebook 04 trains the adapter using the examples from notebook 02. Exit the TUI with `/exit` when done.